In [1]:
from pathlib import Path
from hashlib import sha256
import pandas as pd
from rdflib import Graph
from rdflib.namespace import RDF
from rdflib import Namespace

# 1. Xác định thư mục dự án
PROJECT_DIR = next(
    (
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "data/ACME_Insurance").exists()
        and (p / "results/ontop").exists()
    ),
    None
)

assert PROJECT_DIR is not None, "Không tìm thấy thư mục dự án"

# 2. Khai báo các file KG hiện tại
ONTOLOGY = PROJECT_DIR / "data/ACME_Insurance/ontology/insurance.ttl"
MAPPING = PROJECT_DIR / "results/ontop/acme_mapping.ttl"

OUTPUT_DIR = PROJECT_DIR / "results/extended_analysis/kg_quality_ablation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for path in [ONTOLOGY, MAPPING]:
    assert path.exists(), f"Thiếu file: {path}"

# 3. Đọc ontology và R2RML mapping
ontology = Graph()
ontology.parse(ONTOLOGY, format="turtle")

mapping = Graph()
mapping.parse(MAPPING, format="turtle")

RR = Namespace("http://www.w3.org/ns/r2rml#")

n_maps = len(set(mapping.subjects(RDF.type, RR.TriplesMap)))

print("Ontology triples:", len(ontology))
print("Mapping triples:", len(mapping))
print("R2RML TriplesMaps:", n_maps)

# 4. Lưu dấu vân tay các file đầu vào
inventory = []

for name, path in [("Ontology", ONTOLOGY), ("Mapping", MAPPING)]:
    content = path.read_bytes()

    inventory.append({
        "Component": name,
        "Path": str(path.relative_to(PROJECT_DIR)),
        "Size (bytes)": len(content),
        "SHA256": sha256(content).hexdigest()
    })

inventory = pd.DataFrame(inventory)
inventory.to_csv(OUTPUT_DIR / "source_inventory.csv", index=False)

display(inventory)

# 5. Kiểm tra những vấn đề metadata đã phát hiện
content = MAPPING.read_text(encoding="utf-8")

checks = {
    "PolicyHolder-": content.count("PolicyHolder-"),
    "Policy-Holder-": content.count("Policy-Holder-"),
    "xsd:date": content.count("xsd:date") - content.count("xsd:dateTime"),
    "xsd:dateTime": content.count("xsd:dateTime"),
    "fn:date_diff": content.count("fn:date_diff"),
    "data.world table prefix": content.count(
        "myinsurancecompany.omg-pc-database"
    )
}

print("\nMetadata occurrences:")
display(pd.DataFrame(
    [{"Pattern": k, "Count": v} for k, v in checks.items()]
))

# 6. Tìm các file TTL nguồn có thể dùng để đối chiếu
source_dir = PROJECT_DIR / "data/ACME_Insurance"
ttl_files = sorted(source_dir.rglob("*.ttl"))

print("\nCác file TTL trong thư mục dữ liệu:")
for path in ttl_files:
    print(path.relative_to(PROJECT_DIR))

print("\nOutput:", OUTPUT_DIR)

Ontology triples: 164
Mapping triples: 561
R2RML TriplesMaps: 62


,Component,Path,Size (bytes),SHA256
0,Ontology,data\ACME_Insurance\ontology\insurance.ttl,12876,485ecb19d00cc2906270ecee4b75dc9cbf55c7c653fdef...
1,Mapping,results\ontop\acme_mapping.ttl,43535,60b01d08394017d75a14a46bd0e9873f9fa50125861beb...



Metadata occurrences:


,Pattern,Count
0,PolicyHolder-,0
1,Policy-Holder-,3
2,xsd:date,0
3,xsd:dateTime,0
4,fn:date_diff,0
5,data.world table prefix,0



Các file TTL trong thư mục dữ liệu:
data\ACME_Insurance\investigation\acme-benchmark.ttl
data\ACME_Insurance\ontology\insurance.ttl

Output: c:\Users\Win 11 New\Desktop\thuc-nghiem-co-so-httt\results\extended_analysis\kg_quality_ablation


In [2]:
import re
from collections import Counter
from rdflib.namespace import RDF, RDFS, OWL, XSD

# 1. Thống kê các thành phần R2RML
rr = Namespace("http://www.w3.org/ns/r2rml#")

stats = {
    "TriplesMaps": len(set(mapping.subjects(RDF.type, rr.TriplesMap))),
    "Logical Tables": len(set(mapping.objects(None, rr.logicalTable))),
    "SQL Queries": len(list(mapping.triples((None, rr.sqlQuery, None)))),
    "Table Names": len(list(mapping.triples((None, rr.tableName, None)))),
    "Subject Maps": len(list(mapping.triples((None, rr.subjectMap, None)))),
    "Predicate-Object Maps": len(list(mapping.triples((None, rr.predicateObjectMap, None)))),
    "Object Maps": len(list(mapping.triples((None, rr.objectMap, None))))
}

print("Cấu trúc R2RML:")
display(pd.DataFrame(
    [{"Component": k, "Count": v} for k, v in stats.items()]
))

# 2. Kiểm tra datatype trong mapping
datatypes = Counter(
    str(obj)
    for obj in mapping.objects(None, rr.datatype)
)

print("\nDatatypes trong mapping:")
display(pd.DataFrame(
    [{"Datatype": k, "Count": v} for k, v in datatypes.items()]
))

# 3. Kiểm tra các vấn đề metadata trong cả ontology và mapping
patterns = {
    "PolicyHolder URI": r"PolicyHolder-",
    "Policy-Holder URI": r"Policy-Holder-",
    "Date datatype": r"(?:xsd:date\b|#date[\"'>])",
    "DateTime datatype": r"(?:xsd:dateTime|#dateTime\b)",
    "Date difference function": r"date[_-]?diff|datediff",
    "Data.world table prefix": r"myinsurancecompany|omg-pc-database"
}

sources = {
    "Ontology": ONTOLOGY.read_text(encoding="utf-8"),
    "Mapping": MAPPING.read_text(encoding="utf-8")
}

checks = []

for component, content in sources.items():
    for name, pattern in patterns.items():
        matches = list(re.finditer(pattern, content, re.IGNORECASE))
        checks.append({
            "Component": component,
            "Issue": name,
            "Occurrences": len(matches)
        })

audit = pd.DataFrame(checks)
display(audit)

# 4. Hiển thị các dòng liên quan để xác minh trực tiếp
print("\nCác dòng metadata cần kiểm tra:")

for component, content in sources.items():
    lines = content.splitlines()

    matched = [
        (i, line.strip())
        for i, line in enumerate(lines, start=1)
        if any(
            re.search(pattern, line, re.IGNORECASE)
            for pattern in patterns.values()
        )
    ]

    print(f"\n{component}: {len(matched)} dòng")

    for line_no, line in matched[:20]:
        print(f"Line {line_no}: {line[:180]}")

# 5. Lưu kết quả kiểm toán
audit.to_csv(
    OUTPUT_DIR / "metadata_audit.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nĐã lưu metadata_audit.csv")


Cấu trúc R2RML:


,Component,Count
0,TriplesMaps,62
1,Logical Tables,62
2,SQL Queries,39
3,Table Names,23
4,Subject Maps,62
5,Predicate-Object Maps,50
6,Object Maps,50



Datatypes trong mapping:


""


,Component,Issue,Occurrences
0,Ontology,PolicyHolder URI,0
1,Ontology,Policy-Holder URI,0
2,Ontology,Date datatype,0
3,Ontology,DateTime datatype,6
4,Ontology,Date difference function,0
5,Ontology,Data.world table prefix,0
6,Mapping,PolicyHolder URI,0
7,Mapping,Policy-Holder URI,3
8,Mapping,Date datatype,0
9,Mapping,DateTime datatype,0



Các dòng metadata cần kiểm tra:

Ontology: 6 dòng
Line 120: rdfs:range xsd:dateTime ;
Line 135: rdfs:range xsd:dateTime ;
Line 171: rdfs:range xsd:dateTime ;
Line 179: rdfs:range xsd:dateTime ;
Line 187: rdfs:range xsd:dateTime ;
Line 195: rdfs:range xsd:dateTime ;

Mapping: 120 dòng
Line 6: rr:template "https://myinsurancecompany.linked.data.world/d/omg-pc-database/Agent-{party_identifier}" ] ;
Line 11: dwo:queryBase <https://myinsurancecompany.linked.data.world/d/omg-pc-database/> ] .
Line 13: rr:predicateObjectMap  [ rr:objectMap [ rr:template "https://myinsurancecompany.linked.data.world/d/omg-pc-database/Claim-{claim_identifier}" ] ;
Line 15: rr:subjectMap          [ rr:template "https://myinsurancecompany.linked.data.world/d/omg-pc-database/PolicyCoverageDetail-{policy_coverage_detail_identifier}" ] ;
Line 18: rr:predicateObjectMap  [ rr:objectMap [ rr:template "https://myinsurancecompany.linked.data.world/d/omg-pc-database/Catastrophe-{catastrophe_identifier}" ] ;
Line 20: rr:s

In [3]:
from rdflib.namespace import RDF, RDFS, XSD

RR = Namespace("http://www.w3.org/ns/r2rml#")

# 1. Tìm các URI Policy Holder trong SubjectMap và ObjectMap
uri_records = []

for tm in mapping.subjects(RDF.type, RR.TriplesMap):
    subject_maps = list(mapping.objects(tm, RR.subjectMap))
    object_maps = []

    for pom in mapping.objects(tm, RR.predicateObjectMap):
        object_maps.extend(mapping.objects(pom, RR.objectMap))

    for role, nodes in [
        ("SubjectMap", subject_maps),
        ("ObjectMap", object_maps)
    ]:
        for node in nodes:
            for template in mapping.objects(node, RR.template):
                value = str(template)

                if "Policy-Holder-" in value or "PolicyHolder-" in value:
                    uri_records.append({
                        "TriplesMap": str(tm),
                        "Role": role,
                        "Template": value
                    })

uri_df = pd.DataFrame(uri_records)

print("Policy Holder URI templates:")
display(uri_df)

# 2. Xác định các thuộc tính có range xsd:dateTime
date_records = []

for prop in ontology.subjects(RDFS.range, XSD.dateTime):
    domains = list(ontology.objects(prop, RDFS.domain))

    date_records.append({
        "Property": str(prop),
        "Domain": ", ".join(map(str, domains)),
        "Range": str(XSD.dateTime)
    })

date_df = pd.DataFrame(date_records)

print("\nCác thuộc tính xsd:dateTime:")
display(date_df)

# 3. Kiểm tra tên bảng và SQL thực tế trong R2RML
sql_records = []

for tm in mapping.subjects(RDF.type, RR.TriplesMap):
    for logical_table in mapping.objects(tm, RR.logicalTable):
        for field, predicate in [
            ("sqlQuery", RR.sqlQuery),
            ("tableName", RR.tableName)
        ]:
            for value in mapping.objects(logical_table, predicate):
                sql_records.append({
                    "TriplesMap": str(tm),
                    "Type": field,
                    "Value": str(value)
                })

sql_df = pd.DataFrame(sql_records)

# Chỉ tìm lỗi trong khai báo SQL, không tính URI template
patterns = {
    "date_diff": r"(?i)date[_-]?diff",
    "Data.world table name": r"(?i)myinsurancecompany\.omg-pc-database",
}

print("\nKiểm tra SQL logical tables:")
print("Tổng khai báo:", len(sql_df))

for name, pattern in patterns.items():
    matches = sql_df["Value"].str.contains(
        pattern, regex=True, na=False
    )
    print(f"{name}: {matches.sum()}")

# 4. Lưu kết quả kiểm toán
uri_df.to_csv(OUTPUT_DIR / "uri_candidates.csv", index=False)
date_df.to_csv(OUTPUT_DIR / "date_candidates.csv", index=False)
sql_df.to_csv(OUTPUT_DIR / "logical_tables_audit.csv", index=False)

print("\nĐã lưu các bảng kiểm toán.")


Policy Holder URI templates:


,TriplesMap,Role,Template
0,http://capsenta.com/mappings#TripleMap_PolicyH...,SubjectMap,https://myinsurancecompany.linked.data.world/d...
1,http://capsenta.com/mappings#TripleMap_PolicyH...,SubjectMap,https://myinsurancecompany.linked.data.world/d...
2,http://capsenta.com/mappings#TripleMap_haspoli...,ObjectMap,https://myinsurancecompany.linked.data.world/d...



Các thuộc tính xsd:dateTime:


,Property,Domain,Range
0,http://data.world/schema/insurance/claimCloseDate,http://data.world/schema/insurance/Claim,http://www.w3.org/2001/XMLSchema#dateTime
1,http://data.world/schema/insurance/claimOpenDate,http://data.world/schema/insurance/Claim,http://www.w3.org/2001/XMLSchema#dateTime
2,http://data.world/schema/insurance/policyCover...,http://data.world/schema/insurance/PolicyCover...,http://www.w3.org/2001/XMLSchema#dateTime
3,http://data.world/schema/insurance/policyCover...,http://data.world/schema/insurance/PolicyCover...,http://www.w3.org/2001/XMLSchema#dateTime
4,http://data.world/schema/insurance/policyEffec...,http://data.world/schema/insurance/Policy,http://www.w3.org/2001/XMLSchema#dateTime
5,http://data.world/schema/insurance/policyExpir...,http://data.world/schema/insurance/Policy,http://www.w3.org/2001/XMLSchema#dateTime



Kiểm tra SQL logical tables:
Tổng khai báo: 62
date_diff: 0
Data.world table name: 0

Đã lưu các bảng kiểm toán.


In [4]:
import shutil
from rdflib import Graph, Literal
from rdflib.namespace import RDF, RDFS, XSD

VARIANTS_DIR = OUTPUT_DIR / "variants"
VARIANTS_DIR.mkdir(parents=True, exist_ok=True)

# 1. Baseline: giữ nguyên các file local
baseline_dir = VARIANTS_DIR / "baseline"
baseline_dir.mkdir(exist_ok=True)

shutil.copy2(ONTOLOGY, baseline_dir / "ontology.ttl")
shutil.copy2(MAPPING, baseline_dir / "mapping.ttl")

# 2. URI mismatch: chỉ thay URI trong một ObjectMap
uri_dir = VARIANTS_DIR / "uri_mismatch"
uri_dir.mkdir(exist_ok=True)

uri_mapping = Graph()
uri_mapping += mapping

changed_uri = 0

for obj_map in mapping.objects(None, RR.objectMap):
    for template in mapping.objects(obj_map, RR.template):
        old = str(template)

        if "Policy-Holder-" not in old:
            continue

        new = old.replace("Policy-Holder-", "PolicyHolder-", 1)

        uri_mapping.remove((obj_map, RR.template, template))
        uri_mapping.add((obj_map, RR.template, Literal(new)))
        changed_uri += 1

assert changed_uri == 1, f"Số ObjectMaps thay đổi: {changed_uri}"
assert len(uri_mapping) == len(mapping)

shutil.copy2(ONTOLOGY, uri_dir / "ontology.ttl")
uri_mapping.serialize(uri_dir / "mapping.ttl", format="turtle")

# 3. Date range: thay đổi 6 khai báo datatype trong ontology
date_dir = VARIANTS_DIR / "date_range"
date_dir.mkdir(exist_ok=True)

date_ontology = Graph()
date_ontology += ontology

date_properties = list(
    ontology.subjects(RDFS.range, XSD.dateTime)
)

assert len(date_properties) == 6

for prop in date_properties:
    date_ontology.remove((prop, RDFS.range, XSD.dateTime))
    date_ontology.add((prop, RDFS.range, XSD.date))

assert len(date_ontology) == len(ontology)

date_ontology.serialize(date_dir / "ontology.ttl", format="turtle")
shutil.copy2(MAPPING, date_dir / "mapping.ttl")

# 4. Kiểm tra các file tạo ra
records = []

for name in ["baseline", "uri_mismatch", "date_range"]:
    folder = VARIANTS_DIR / name

    ont = Graph()
    ont.parse(folder / "ontology.ttl", format="turtle")

    mp = Graph()
    mp.parse(folder / "mapping.ttl", format="turtle")

    n_maps = len(set(mp.subjects(RDF.type, RR.TriplesMap)))

    assert len(ont) == 164
    assert len(mp) == 561
    assert n_maps == 62

    records.append({
        "Variant": name,
        "Ontology Triples": len(ont),
        "Mapping Triples": len(mp),
        "TriplesMaps": n_maps,
        "Valid": True
    })

variant_summary = pd.DataFrame(records)

variant_summary.to_csv(
    OUTPUT_DIR / "variant_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

display(variant_summary)
print("URI templates modified:", changed_uri)
print("Date properties modified:", len(date_properties))
print("Output:", VARIANTS_DIR)


,Variant,Ontology Triples,Mapping Triples,TriplesMaps,Valid
0,baseline,164,561,62,True
1,uri_mismatch,164,561,62,True
2,date_range,164,561,62,True


URI templates modified: 1
Date properties modified: 6
Output: c:\Users\Win 11 New\Desktop\thuc-nghiem-co-so-httt\results\extended_analysis\kg_quality_ablation\variants


In [5]:
import subprocess
import time

# 1. Cấu hình Ontop
ONTOP = Path.home() / "tools/ontop/ontop.bat"
PROPERTIES = PROJECT_DIR / "results/ontop/duckdb.properties"

# Dùng Q01 đã thực thi thành công ở Context Ablation
TEST_QUERY = (
    PROJECT_DIR
    / "results/extended_analysis/context_ablation"
    / "evaluation/sparql_results/run_01/01_ontop.rq"
)

assert ONTOP.exists()
assert PROPERTIES.exists()
assert TEST_QUERY.exists(), f"Thiếu truy vấn: {TEST_QUERY}"

SMOKE_DIR = OUTPUT_DIR / "smoke_test"
SMOKE_DIR.mkdir(exist_ok=True)

# 2. Kiểm tra từng biến thể
records = []

for variant in ["baseline", "uri_mismatch", "date_range"]:
    folder = VARIANTS_DIR / variant
    mapping_file = folder / "mapping.ttl"
    ontology_file = folder / "ontology.ttl"
    output_file = SMOKE_DIR / f"{variant}.csv"
    log_file = SMOKE_DIR / f"{variant}.log"

    output_file.unlink(missing_ok=True)
    start = time.perf_counter()

    try:
        # Sử dụng cách truyền đường dẫn đã kiểm chứng
        cmd = [
            str(ONTOP), "query",
            "-m", mapping_file.resolve().as_uri(),
            "-t", str(ontology_file.resolve()),
            "-p", PROPERTIES.resolve().as_uri(),
            "-q", str(TEST_QUERY.resolve()),
            "-o", str(output_file)
        ]

        result = subprocess.run(
            cmd,
            cwd=ONTOP.parent,
            capture_output=True,
            text=True,
            encoding="utf-8",
            errors="replace",
            timeout=120
        )

        log = result.stderr + "\n" + result.stdout
        log_file.write_text(log, encoding="utf-8")

        # Ontop đôi khi trả exit code 0 dù thực thi lỗi
        if result.returncode != 0 or not output_file.exists():
            raise RuntimeError(log[-1500:] or "Không tạo CSV")

        if "RepositoryException" in log or "Exception in thread" in log:
            raise RuntimeError(log[-1500:])

        df = pd.read_csv(output_file)

        status = "OK"
        rows = len(df)
        error = ""

    except Exception as e:
        status = "ERROR"
        rows = 0
        error = str(e)[:500]

    records.append({
        "Variant": variant,
        "Status": status,
        "Rows": rows,
        "Time (s)": round(time.perf_counter() - start, 2),
        "Error": error
    })

# 3. Lưu và hiển thị kết quả
smoke = pd.DataFrame(records)

smoke.to_csv(
    OUTPUT_DIR / "smoke_test.csv",
    index=False,
    encoding="utf-8-sig"
)

display(smoke)

assert smoke["Status"].eq("OK").all(), \
    "Có biến thể chưa chạy được. Kiểm tra log trong smoke_test."

print("Hoàn thành kiểm tra 3 biến thể.")


,Variant,Status,Rows,Time (s),Error
0,baseline,OK,2,5.58,
1,uri_mismatch,OK,2,5.45,
2,date_range,OK,2,5.44,


Hoàn thành kiểm tra 3 biến thể.


In [14]:

import re
import time
import hashlib
import subprocess
import pandas as pd
from rdflib import Graph, Namespace
from rdflib.namespace import RDF
from pathlib import Path

# 1. Đọc benchmark gốc
BENCHMARK_FILE = (
    PROJECT_DIR / "data/ACME_Insurance/investigation/acme-benchmark.ttl"
)

benchmark = Graph()
benchmark.parse(BENCHMARK_FILE, format="turtle")

QA = Namespace("http://models.data.world/benchmarks/QandA#")
DWT = Namespace("https://templates.data.world/")

extra_question = (
    "Return all the claims by claim number "
    "that were closed in 2019"
)

inquiries = sorted(
    set(benchmark.subjects(RDF.type, QA.Inquiry)),
    key=str
)

inquiries = [
    iq for iq in inquiries
    if str(benchmark.value(iq, QA.prompt)) != extra_question
]

assert len(inquiries) == 43


# 2. Trích xuất 43 SPARQL Reference
references = []


# Nhận diện SERVICE <URI>, SERVICE prefix:mapped và SERVICE :mapped
service_pattern = re.compile(
    r"\bSERVICE\s+(?:SILENT\s+)?"
    r"(?P<target><[^>]+>|(?:[A-Za-z_][\w.-]*)?:[\w.-]+)"
    r"\s*(?=\{)",
    flags=re.IGNORECASE
)

# Kiểm tra các cú pháp SERVICE
for example in [
    "SERVICE :mapped {",
    "SERVICE <mapped> {",
    "SERVICE <https://myinsurancecompany.linked.data.world/d/chat-with-the-data-benchmark/mapped> {"
]:
    assert service_pattern.search(example), example

print("Đã nhận diện đầy đủ 3 dạng SERVICE")


for qid, inquiry in enumerate(inquiries, start=1):
    candidates = []

    for ref in benchmark.objects(inquiry, QA.expects):
        types = set(benchmark.objects(ref, RDF.type))
        languages = [
            str(x).lower()
            for x in benchmark.objects(ref, QA.inLanguage)
        ]

        is_sparql = (
            DWT.SparqlQuery in types
            or any("sparql" in x for x in languages)
        )

        if not is_sparql:
            continue

        query_text = benchmark.value(ref, QA.queryText)

        if query_text is not None and str(query_text).strip():
            candidates.append((str(ref), str(query_text)))

    assert candidates, f"Q{qid:02d} không có SPARQL Reference"

    ref_uri, query = sorted(
        candidates, key=lambda x: x[0]
    )[0]

    # Loại bỏ SERVICE wrapper của data.world
    matches = list(service_pattern.finditer(query))

    for match in reversed(matches):
        target = match.group("target")

        if "mapped" not in target.lower():
            raise ValueError(
                f"Q{qid:02d}: SERVICE không thuộc KG local: {target}"
            )

        query = query[:match.start()] + query[match.end():]

    query = query.strip()

    # Nếu còn SERVICE, in đoạn gốc để chẩn đoán
    remaining = re.search(r"\bSERVICE\b", query, re.IGNORECASE)

    if remaining:
        pos = remaining.start()
        snippet = query[max(0, pos - 50):pos + 180]

        raise ValueError(
            f"Q{qid:02d} vẫn chứa SERVICE: {snippet!r}"
        )

    references.append({
        "ID": qid,
        "Reference URI": ref_uri,
        "Query": query,
        "Reference Count": len(candidates)
    })

ref_df = pd.DataFrame(references)

assert len(ref_df) == 43

print("Đã xử lý:", len(ref_df), "SPARQL References")
print("Q01 sau xử lý:")
print(ref_df.iloc[0]["Query"][:500])


# 3. Kiểm tra và lưu 43 SPARQL Reference
assert len(ref_df) == 43
assert ref_df["ID"].nunique() == 43

RUN_DIR = OUTPUT_DIR / "reference_execution"
RUN_DIR.mkdir(parents=True, exist_ok=True)

ref_df.to_csv(
    OUTPUT_DIR / "selected_references.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Đã lưu 43 SPARQL Reference")


# 4. Đọc checkpoint cũ
DETAIL_FILE = OUTPUT_DIR / "reference_execution_detail.csv"

if DETAIL_FILE.exists():
    existing = pd.read_csv(
        DETAIL_FILE,
        keep_default_na=False
    )
    records = existing.to_dict("records")
else:
    records = []

completed = {
    (str(r["Variant"]), int(r["ID"])): r
    for r in records
}

# 5. Thực thi 43 Reference trên 3 biến thể
variants = ["baseline", "uri_mismatch", "date_range"]

for variant in variants:
    folder = VARIANTS_DIR / variant

    mapping_file = folder / "mapping.ttl"
    ontology_file = folder / "ontology.ttl"

    variant_dir = RUN_DIR / variant
    variant_dir.mkdir(parents=True, exist_ok=True)

    mapping_hash = hashlib.sha256(
        mapping_file.read_bytes()
    ).hexdigest()

    ontology_hash = hashlib.sha256(
        ontology_file.read_bytes()
    ).hexdigest()

    print(f"\nVariant: {variant}")

    for row in ref_df.itertuples(index=False):
        qid = int(row.ID)
        query = row.Query

        key = (variant, qid)

        query_hash = hashlib.sha256(
            query.encode("utf-8")
        ).hexdigest()

        query_file = variant_dir / f"{qid:02d}.rq"
        output_file = variant_dir / f"{qid:02d}.csv"
        log_file = variant_dir / f"{qid:02d}.log"

        # Chỉ bỏ qua kết quả OK nếu cấu hình và file kết quả không đổi
        if key in completed:
            old = completed[key]

            if (
                old.get("Status") == "OK"
                and old.get("Query SHA256") == query_hash
                and old.get("Mapping SHA256") == mapping_hash
                and old.get("Ontology SHA256") == ontology_hash
                and output_file.exists()
            ):
                continue

            # Xóa bản ghi lỗi hoặc bản ghi không còn hợp lệ
            records = [
                r for r in records
                if (str(r["Variant"]), int(r["ID"])) != key
            ]

        query_file.write_text(query, encoding="utf-8")
        output_file.unlink(missing_ok=True)

        status = "ERROR"
        rows = 0
        error = ""
        start = time.perf_counter()

        try:
            # Cấu hình đường dẫn đã kiểm chứng ở Cell 5
            cmd = [
                str(ONTOP), "query",
                "-m", mapping_file.resolve().as_uri(),
                "-t", str(ontology_file.resolve()),
                "-p", PROPERTIES.resolve().as_uri(),
                "-q", str(query_file.resolve()),
                "-o", str(output_file)
            ]

            result = subprocess.run(
                cmd,
                cwd=ONTOP.parent,
                capture_output=True,
                text=True,
                encoding="utf-8",
                errors="replace",
                timeout=120
            )

            log = result.stderr + "\n" + result.stdout
            log_file.write_text(log, encoding="utf-8")

            # Ontop có thể trả exit code 0 dù phát sinh exception
            if result.returncode != 0 or not output_file.exists():
                raise RuntimeError(
                    log[-1500:] or "Ontop không tạo CSV"
                )

            if (
                "RepositoryException" in log
                or "Exception in thread" in log
                or "QueryEvaluationException" in log
            ):
                raise RuntimeError(log[-1500:])

            result_df = pd.read_csv(output_file)

            rows = len(result_df)
            status = "OK"

        except Exception as e:
            error = f"{type(e).__name__}: {e}"[:1500]
            output_file.unlink(missing_ok=True)

        # Lưu kết quả từng truy vấn
        records.append({
            "Variant": variant,
            "ID": qid,
            "Reference URI": row[1],
            "Status": status,
            "Rows": rows,
            "Non-empty": int(status == "OK" and rows > 0),
            "Time (s)": round(
                time.perf_counter() - start, 3
            ),
            "Error": error,
            "Query SHA256": query_hash,
            "Mapping SHA256": mapping_hash,
            "Ontology SHA256": ontology_hash
        })

        completed[key] = records[-1]

        # Lưu checkpoint sau mỗi truy vấn
        pd.DataFrame(records).to_csv(
            DETAIL_FILE,
            index=False,
            encoding="utf-8-sig"
        )

        print(
            f"Q{qid:02d}: {status}, "
            f"Rows={rows}"
        )

# 6. Kiểm tra kết quả và tổng hợp
execution = pd.DataFrame(records)

assert len(execution) == 129, (
    f"Thiếu kết quả: {len(execution)}/129"
)

assert not execution.duplicated(
    ["Variant", "ID"]
).any()

summary = execution.groupby("Variant").agg(
    Queries=("ID", "count"),
    Executed=("Status", lambda x: (x == "OK").sum()),
    Non_empty=("Non-empty", "sum"),
    Errors=("Status", lambda x: (x != "OK").sum())
).reset_index()

summary.to_csv(
    OUTPUT_DIR / "execution_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

display(summary)

print("\nĐã thực thi:", len(execution), "/129 truy vấn")
print("Kết quả:", DETAIL_FILE)

Đã nhận diện đầy đủ 3 dạng SERVICE
Đã xử lý: 43 SPARQL References
Q01 sau xử lý:
PREFIX : <https://myinsurancecompany.linked.data.world/d/chat-with-the-data-benchmark/>
PREFIX in: <http://data.world/schema/insurance/>

SELECT  
  ?policynumber  
  ?agentId
(SUM(?loss) / ?premium AS ?lossRatio)
WHERE {
{
    ?pcd a in:PolicyCoverageDetail;
        in:hasPremiumAmount [ 
            in:premiumAmount ?premium;
        ];
        in:hasPolicy ?policy. 

     ?policy a in:Policy;
            in:policyNumber ?policynumber;
            in:hasPolicyHolder [ 
                in:polic
Đã lưu 43 SPARQL Reference

Variant: baseline
Q01: OK, Rows=1
Q02: OK, Rows=2
Q03: OK, Rows=2
Q04: OK, Rows=1
Q05: OK, Rows=2
Q06: OK, Rows=1
Q07: OK, Rows=2
Q08: OK, Rows=2
Q09: OK, Rows=2
Q10: OK, Rows=2
Q11: OK, Rows=1
Q12: OK, Rows=2
Q13: OK, Rows=2
Q14: OK, Rows=6
Q15: OK, Rows=2
Q16: OK, Rows=2
Q17: OK, Rows=2
Q18: ERROR, Rows=0
Q19: OK, Rows=1
Q20: OK, Rows=2
Q21: OK, Rows=1
Q22: OK, Rows=6
Q23: OK, Rows=6
Q

,Variant,Queries,Executed,Non_empty,Errors
0,baseline,43,42,42,1
1,date_range,43,42,42,1
2,uri_mismatch,43,42,30,1



Đã thực thi: 129 /129 truy vấn
Kết quả: c:\Users\Win 11 New\Desktop\thuc-nghiem-co-so-httt\results\extended_analysis\kg_quality_ablation\reference_execution_detail.csv


In [15]:
import csv
from collections import Counter
import pandas as pd

# 1. Đọc kết quả Cell 6
detail = pd.read_csv(
    OUTPUT_DIR / "reference_execution_detail.csv",
    keep_default_na=False
)

assert len(detail) == 129
assert not detail.duplicated(["Variant", "ID"]).any()

detail["ID"] = detail["ID"].astype(int)
detail["Rows"] = pd.to_numeric(detail["Rows"])
detail["Non-empty"] = pd.to_numeric(detail["Non-empty"])

variants = ["baseline", "uri_mismatch", "date_range"]

for variant in variants:
    assert detail.loc[detail["Variant"] == variant, "ID"].nunique() == 43

# 2. So sánh nội dung CSV, không phụ thuộc thứ tự dòng/cột
def read_result(path):
    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.reader(f)
        header = next(reader)
        rows = list(reader)

    assert len(set(header)) == len(header), "Trùng tên cột"
    assert all(len(row) == len(header) for row in rows)

    order = sorted(range(len(header)), key=lambda i: header[i])
    columns = tuple(header[i] for i in order)
    values = Counter(
        tuple(row[i] for i in order)
        for row in rows
    )

    return columns, values


def same_output(path_a, path_b):
    return read_result(path_a) == read_result(path_b)


# 3. Đối chiếu từng câu hỏi với baseline
lookup = {
    (row["Variant"], int(row["ID"])): row
    for _, row in detail.iterrows()
}

comparison = []

for variant in variants:
    for qid in range(1, 44):
        base = lookup[("baseline", qid)]
        current = lookup[(variant, qid)]

        base_ok = base["Status"] == "OK"
        current_ok = current["Status"] == "OK"

        if not base_ok:
            match = None
            comparison_status = "BASELINE_ERROR"

        elif not current_ok:
            match = None
            comparison_status = "VARIANT_ERROR"

        else:
            base_file = (
                OUTPUT_DIR / "reference_execution"
                / "baseline" / f"{qid:02d}.csv"
            )
            variant_file = (
                OUTPUT_DIR / "reference_execution"
                / variant / f"{qid:02d}.csv"
            )

            assert base_file.exists(), f"Thiếu {base_file}"
            assert variant_file.exists(), f"Thiếu {variant_file}"

            match = same_output(base_file, variant_file)
            comparison_status = "SAME" if match else "CHANGED"

        comparison.append({
            "Variant": variant,
            "ID": qid,
            "Baseline Status": base["Status"],
            "Variant Status": current["Status"],
            "Baseline Rows": int(base["Rows"]),
            "Variant Rows": int(current["Rows"]),
            "Match Baseline": match,
            "Comparison Status": comparison_status
        })

comparison = pd.DataFrame(comparison)

# 4. Tổng hợp theo biến thể
summary = []

for variant in variants:
    d = detail[detail["Variant"] == variant]
    c = comparison[comparison["Variant"] == variant]

    comparable = c["Comparison Status"].isin(["SAME", "CHANGED"])

    summary.append({
        "Variant": variant,
        "Queries": len(d),
        "Executed": int(d["Status"].eq("OK").sum()),
        "Non-empty": int(d["Non-empty"].sum()),
        "Errors": int(d["Status"].ne("OK").sum()),
        "Comparable": int(comparable.sum()),
        "Same as Baseline": int(c["Comparison Status"].eq("SAME").sum()),
        "Changed": int(c["Comparison Status"].eq("CHANGED").sum())
    })

summary = pd.DataFrame(summary)

# 5. Lưu kết quả
comparison.to_csv(
    OUTPUT_DIR / "result_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

summary.to_csv(
    OUTPUT_DIR / "kg_quality_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

errors = detail[detail["Status"] != "OK"].copy()

errors.to_csv(
    OUTPUT_DIR / "execution_errors.csv",
    index=False,
    encoding="utf-8-sig"
)

# 6. Hiển thị
print("KG Quality Ablation Summary:")
display(summary)

print("\nCác truy vấn thực thi lỗi:")
display(errors[["Variant", "ID", "Status", "Error"]])

print("\nCác truy vấn thay đổi kết quả:")
display(
    comparison.loc[
        comparison["Comparison Status"] == "CHANGED",
        ["Variant", "ID", "Baseline Rows", "Variant Rows"]
    ]
)

assert len(comparison) == 129

print("\nĐã đối chiếu 129 bản ghi.")
print("Kết quả lưu tại:", OUTPUT_DIR)


KG Quality Ablation Summary:


,Variant,Queries,Executed,Non-empty,Errors,Comparable,Same as Baseline,Changed
0,baseline,43,42,42,1,42,42,0
1,uri_mismatch,43,42,30,1,42,30,12
2,date_range,43,42,42,1,42,42,0



Các truy vấn thực thi lỗi:


,Variant,ID,Status,Error
17,baseline,18,ERROR,RuntimeError: RDF4JTupleExprTranslator.java:17...
60,uri_mismatch,18,ERROR,RuntimeError: RDF4JTupleExprTranslator.java:17...
103,date_range,18,ERROR,RuntimeError: RDF4JTupleExprTranslator.java:17...



Các truy vấn thay đổi kết quả:


,Variant,ID,Baseline Rows,Variant Rows
43,uri_mismatch,1,1,0
45,uri_mismatch,3,2,0
49,uri_mismatch,7,2,0
50,uri_mismatch,8,2,0
51,uri_mismatch,9,2,0
56,uri_mismatch,14,6,0
66,uri_mismatch,24,2,0
68,uri_mismatch,26,1,0
69,uri_mismatch,27,1,0
73,uri_mismatch,31,1,0



Đã đối chiếu 129 bản ghi.
Kết quả lưu tại: c:\Users\Win 11 New\Desktop\thuc-nghiem-co-so-httt\results\extended_analysis\kg_quality_ablation
